In [ ]:
from pathlib import Path

import py3Dmol
from rdkit import Chem

from docking import ligand, receptor, docking, affinity, view

from ipywidgets import interact, IntSlider
import ipywidgets, copy

work_dir = Path("data")
work_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
Navitoclax = ligand(work_dir, "CC1(CCC(=C(C1)CN2CCN(CC2)C3=CC=C(C=C3)C(=O)NS(=O)(=O)C4=CC(=C(C=C4)N[C@H](CCN5CCOCC5)CSC6=CC=CC=C6)S(=O)(=O)C(F)(F)F)C7=CC=C(C=C7)Cl)C", 7.4)

In [ ]:
v = py3Dmol.view()
v.addModel(open(view(work_dir, Navitoclax), 'r').read(), 'sdf')
v.zoomTo()
v.setBackgroundColor('white')
v.addStyle({'stick': {'colorscheme':'yellowCarbon'}})
v.show()

In [ ]:
Bcl_xL = receptor(work_dir, "2YXJ", (-9, -16, 11), (25, 31, 18))

In [ ]:
def Receptor3DView(receptorPDB, boxPDB):
    v = py3Dmol.view()
    v.setBackgroundColor('white')
    
    v.addModel(open(boxPDB, 'r').read(),'pdb')
    v.addStyle({'stick': {}})
    v.zoomTo()
    
    v.addModel(open(receptorPDB, 'r').read(), 'pdb')
    v.addStyle({'cartoon': {'color':'spectrum', 'opacity': 0.5}})

    return v

Receptor3DView(view(work_dir, Bcl_xL), view(work_dir, Bcl_xL).replace('.pdb', '.box.pdb')).show()

In [ ]:
Bcl_xL_Navitoclax = docking(work_dir, Bcl_xL, Navitoclax, 32)

In [ ]:
affinity(work_dir, Bcl_xL_Navitoclax)

In [ ]:
def Complex3DView(view, ligmol = None, refligPDB = None, reflig_resn = None):

    new_viewer = copy.deepcopy(view)

    mblock = Chem.MolToMolBlock(ligmol)
    new_viewer.addModel(mblock, 'mol')
    new_viewer.addStyle({'hetflag': True}, {"stick": {'colorscheme': 'greenCarbon'}})

    return new_viewer

confs = Chem.SDMolSupplier(view(work_dir, Bcl_xL_Navitoclax))

def conf_viewer(idx):
    mol = confs[idx]
    return Complex3DView(Receptor3DView(receptorPDB = view(work_dir, Bcl_xL), boxPDB = view(work_dir, Bcl_xL).replace('.pdb', '.box.pdb')), mol).show()

interact(conf_viewer, idx=ipywidgets.IntSlider(min=0, max=len(confs)-1, step=1))